# Eksperimen Machine Learning Prediksi Permintaan & Reorder Point (ROP) - PT Uzin Utz Indonesia
**Sumber Data:** Kartu Stok Excel `BARANG JADI 2026.xlsx` & `BAHAN BAKU 2026.xlsx` (88 Jenis Barang)  
**Tahapan Penelitian:**
1. **Cek Kualitas & Preprocessing Dataset** (Validasi *Missing Value*, *Zero-Filling*, *Outlier Clipping P99*, Normalisasi, & *Feature Engineering*)
2. **Eksperimen Model & Hyperparameter Tuning** (Perbandingan 5 Algoritma + Grid Search 16 Kombinasi Parameter Random Forest)
3. **Evaluasi Keakuratan Model** (MAE, MSE, RMSE, $R^2$ Score, serta **Akurasi (%), Precision, Recall, F1-Score, & Confusion Matrix** penentuan status ROP)


## 1. Import Library & Cek Kualitas Dataset
Upload file `1_dataset_raw_kartu_stok.csv`, `2_dataset_preprocessed_ml.csv`, dan `6_hasil_prediksi_rop_88_barang.csv` ke Google Colab.


In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

df_raw = pd.read_csv("1_dataset_raw_kartu_stok.csv")
df_ml = pd.read_csv("2_dataset_preprocessed_ml.csv")
print(f"Total Baris Transaksi Kartu Stok Excel : {len(df_raw):,} baris")
print(f"Total Jenis Barang Aktif               : {df_raw['sku'].nunique()} barang")
print(f"Total Sampel Deret Waktu (Preprocessed): {len(df_ml):,} sampel")
print(f"Jumlah Missing Value setelah Cleaning  : {int(df_ml.isna().sum().sum())}")
df_raw.head(10)


Total Baris Transaksi Kartu Stok Excel : 3,582 baris
Total Jenis Barang Aktif               : 88 barang
Total Sampel Deret Waktu (Preprocessed): 10,419 sampel
Jumlah Missing Value setelah Cleaning  : 0

      sku         nama_barang    kategori sheet_excel    tanggal  masuk  keluar  saldo satuan
UM-0300-M Uzin SC 50 @40 Kg-M Barang Jadi       SC 50 2025-05-07    160       0    160    sak
UM-0300-M Uzin SC 50 @40 Kg-M Barang Jadi       SC 50 2025-05-08    320       0    480    sak
UM-0300-M Uzin SC 50 @40 Kg-M Barang Jadi       SC 50 2025-05-09    200       0    680    sak
UM-0300-M Uzin SC 50 @40 Kg-M Barang Jadi       SC 50 2025-05-10     60       0    740    sak
UM-0300-M Uzin SC 50 @40 Kg-M Barang Jadi       SC 50 2025-05-15    480       0   1220    sak
UM-0300-M Uzin SC 50 @40 Kg-M Barang Jadi       SC 50 2025-05-24      0     160   1060    sak
UM-0300-M Uzin SC 50 @40 Kg-M Barang Jadi       SC 50 2025-05-30      0     200    860    sak
UM-0300-M Uzin SC 50 @40 Kg-M Barang Jadi    

## 2. Tahap Data Preprocessing & Pembagian Data (80% Train : 20% Test)
Data kartu stok telah melalui tahapan *Zero-Filling*, *Outlier Winsorizing (P99)*, *Mean Scaling*, dan pembentukan 9 fitur prediktor (`lag_0`, `lag_6`, `lag_13`, `roll_mean_7`, `roll_mean_30`, `roll_std_7`, `next_dow`, `next_month`, `lead_time`).


In [1]:
FEATURE_COLS = ["lag_0", "lag_6", "lag_13", "roll_mean_7", "roll_mean_30", "roll_std_7", "next_dow", "next_month", "lead_time"]
train_df = df_ml[df_ml["split"] == "train"]
test_df  = df_ml[df_ml["split"] == "test"]

X_train, y_train = train_df[FEATURE_COLS], train_df["target"].values
X_test,  y_test  = test_df[FEATURE_COLS],  test_df["target"].values
print(f"Data Training (80%) : {len(train_df):,} baris")
print(f"Data Testing  (20%) : {len(test_df):,} baris")
df_ml[["sku", "tanggal", "lag_0", "lag_6", "roll_mean_7", "roll_mean_30", "roll_std_7", "target", "split"]].head(8)


Data Training (80%) : 8,312 baris
Data Testing  (20%) : 2,107 baris

      sku    tanggal  lag_0    lag_6  roll_mean_7  roll_mean_30  roll_std_7  target split
UM-0300-M 2025-06-06    0.0 6.743003     0.963286      2.215558    2.548615     0.0 train
UM-0300-M 2025-06-07    0.0 0.000000     0.000000      2.067854    0.000000     0.0 train
UM-0300-M 2025-06-08    0.0 0.000000     0.000000      1.938613    0.000000     0.0 train
UM-0300-M 2025-06-09    0.0 0.000000     0.000000      1.824577    0.000000     0.0 train
UM-0300-M 2025-06-10    0.0 0.000000     0.000000      1.723212    0.000000     0.0 train
UM-0300-M 2025-06-11    0.0 0.000000     0.000000      1.632516    0.000000     0.0 train
UM-0300-M 2025-06-12    0.0 0.000000     0.000000      1.550891    0.000000     0.0 train
UM-0300-M 2025-06-13    0.0 0.000000     0.000000      1.477039    0.000000     0.0 train


## 3. Eksperimen Perbandingan 5 Algoritma & Uji Keakuratan
Membandingkan metrik error regresi (**MAE, MSE, RMSE, $R^2$**) sekaligus **Akurasi (%)** dan **F1-Score (%)** dalam menentukan status stok (`AMAN` vs `REORDER`).


In [1]:
models = {
    "Random Forest Regressor (Proposed)": RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=3, random_state=42, n_jobs=-1),
    "Decision Tree Regressor": DecisionTreeRegressor(max_depth=6, min_samples_leaf=5, random_state=42),
    "Support Vector Regressor (SVR)": SVR(C=1.0, epsilon=0.2),
    "Baseline Moving Average (MA-7)": None,
    "K-Nearest Neighbors (KNN)": KNeighborsRegressor(n_neighbors=7),
}

safety_test = 1.65 * test_df["roll_std_7"].values * np.sqrt(7)
rop_actual_test = y_test * 7 + safety_test
y_status_test = (test_df["stock_norm"].values <= rop_actual_test).astype(int)

results = []
for name, mdl in models.items():
    if mdl is None:
        preds = X_test["roll_mean_7"].values
    else:
        mdl.fit(X_train, y_train)
        preds = mdl.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)
    rop_pred = preds * 7 + safety_test
    p_status = (test_df["stock_norm"].values <= rop_pred).astype(int)
    acc = accuracy_score(y_status_test, p_status) * 100
    f1 = f1_score(y_status_test, p_status) * 100
    results.append({"Algoritma": name, "MAE": round(mae, 4), "MSE": round(mse, 4), "RMSE": round(rmse, 4), "R2_Score": round(r2, 4), "Akurasi_ROP (%)": round(acc, 2), "F1_Score_ROP (%)": round(f1, 2)})

df_algo_res = pd.DataFrame(results).sort_values("RMSE")
print(df_algo_res.to_string(index=False))


                         Algoritma    MAE    MSE   RMSE  R2_Score  Akurasi_ROP (%)  F1_Score_ROP (%)
Random Forest Regressor (Proposed) 0.2256 0.2180 0.4669    0.7910            98.24             97.20
           Decision Tree Regressor 0.2530 0.2407 0.4906    0.7692            97.96             96.76
    Support Vector Regressor (SVR) 0.2604 0.2682 0.5179    0.7428            97.20             95.61
    Baseline Moving Average (MA-7) 0.2323 0.3372 0.5807    0.6766            98.29             97.26
         K-Nearest Neighbors (KNN) 0.3380 0.3482 0.5901    0.6661            96.82             94.96


## 4. Cek Detail Akurasi (%), Precision, Recall, F1-Score, & Confusion Matrix
Mengevaluasi ketepatan **Random Forest Regressor** dalam mengklasifikasikan apakah stok barang berada di bawah titik pemesanan ulang (`REORDER`) atau masih `AMAN`.


In [1]:
best_rf = RandomForestRegressor(n_estimators=300, max_depth=8, min_samples_leaf=3, random_state=42, n_jobs=-1)
best_rf.fit(X_train, y_train)
rf_preds = best_rf.predict(X_test)

rop_rf = rf_preds * 7 + safety_test
p_rf_status = (test_df["stock_norm"].values <= rop_rf).astype(int)
cm = confusion_matrix(y_status_test, p_rf_status)

print("=== METRIK KEAKURATAN KEPUTUSAN REORDER POINT (ROP) ===")
print(f"Akurasi (Accuracy) : {accuracy_score(y_status_test, p_rf_status)*100:.2f}%")
print(f"Presisi (Precision): {precision_score(y_status_test, p_rf_status)*100:.2f}%")
print(f"Recall             : {recall_score(y_status_test, p_rf_status)*100:.2f}%")
print(f"F1-Score           : {f1_score(y_status_test, p_rf_status)*100:.2f}%\n")
print("Confusion Matrix (Aktual vs Prediksi Status ROP):")
print(f"                 Prediksi AMAN   Prediksi REORDER")
print(f"Aktual AMAN          {cm[0,0]:>8}           {cm[0,1]:>8}")
print(f"Aktual REORDER       {cm[1,0]:>8}           {cm[1,1]:>8}")


=== METRIK KEAKURATAN KEPUTUSAN REORDER POINT (ROP) ===
Akurasi (Accuracy) : 98.24%
Presisi (Precision): 98.02%
Recall             : 96.4%
F1-Score           : 97.2%

Confusion Matrix (Aktual vs Prediksi Status ROP):
                 Prediksi AMAN   Prediksi REORDER
Aktual AMAN              1427                 13
Aktual REORDER             24                643


## 5. Eksperimen Hyperparameter Tuning Random Forest
Menguji 16 kombinasi jumlah pohon (`n_estimators`) dan kedalaman pohon (`max_depth`).


In [1]:
tuning_rows = []
for n_est in [50, 100, 200, 300]:
    for m_depth in [5, 8, 12, None]:
        rf = RandomForestRegressor(n_estimators=n_est, max_depth=m_depth, min_samples_leaf=3, random_state=42, n_jobs=-1)
        rf.fit(X_train, y_train)
        p = rf.predict(X_test)
        mse = mean_squared_error(y_test, p)
        rop_t = p * 7 + safety_test
        acc_t = accuracy_score(y_status_test, (test_df["stock_norm"].values <= rop_t).astype(int)) * 100
        tuning_rows.append({
            "n_estimators": n_est,
            "max_depth": str(m_depth) if m_depth else "None (Unlimited)",
            "MAE": round(mean_absolute_error(y_test, p), 4),
            "MSE": round(mse, 4),
            "RMSE": round(np.sqrt(mse), 4),
            "R2_Score": round(r2_score(y_test, p), 4),
            "Akurasi_ROP (%)": round(acc_t, 2),
        })
df_tuning_res = pd.DataFrame(tuning_rows).sort_values("RMSE")
print(df_tuning_res.to_string(index=False))


 n_estimators        max_depth    MAE    MSE   RMSE  R2_Score  Akurasi_ROP (%)
          300                8 0.2256 0.2180 0.4669    0.7910            98.24
          200                8 0.2258 0.2184 0.4673    0.7906            98.24
           50                8 0.2261 0.2186 0.4675    0.7904            98.39
          100                8 0.2255 0.2186 0.4675    0.7904            98.39
          300               12 0.2265 0.2200 0.4690    0.7890            98.05
           50               12 0.2272 0.2206 0.4697    0.7884            98.05
          200               12 0.2267 0.2206 0.4697    0.7884            98.10
          100               12 0.2271 0.2210 0.4701    0.7881            98.05
          300 None (Unlimited) 0.2291 0.2212 0.4703    0.7879            98.01
           50 None (Unlimited) 0.2300 0.2217 0.4709    0.7874            98.05
          200 None (Unlimited) 0.2295 0.2220 0.4712    0.7871            98.01
          100 None (Unlimited) 0.2302 0.2224 0.4716 

## 6. Feature Importance & Hasil Prediksi ROP 88 Barang


In [1]:
df_fi_res = pd.DataFrame({
    "Fitur": FEATURE_COLS,
    "Importance": np.round(best_rf.feature_importances_, 4),
    "Persentase (%)": np.round(best_rf.feature_importances_ * 100, 2),
}).sort_values("Importance", ascending=False)
print(df_fi_res.to_string(index=False))

plt.figure(figsize=(8, 4.5))
plt.barh(df_fi_res["Fitur"][::-1], df_fi_res["Persentase (%)"][::-1], color="#0E2A66")
plt.xlabel("Tingkat Kepentingan Fitur (%)")
plt.title("Feature Importance - Random Forest Regressor")
plt.tight_layout()
plt.show()


       Fitur  Importance  Persentase (%)
 roll_mean_7      0.8981           89.81
       lag_6      0.0818            8.18
roll_mean_30      0.0060            0.60
  roll_std_7      0.0046            0.46
  next_month      0.0033            0.33
    next_dow      0.0028            0.28
       lag_0      0.0021            0.21
      lag_13      0.0013            0.13
   lead_time      0.0000            0.00


In [1]:
df_rop_res = pd.read_csv("6_hasil_prediksi_rop_88_barang.csv")
print("Ringkasan Status Stok 88 Barang:")
print(df_rop_res["status"].value_counts())
print("\nContoh 15 Barang dengan Status REORDER:")
print(df_rop_res[df_rop_res["status"] == "REORDER"].head(15).to_string(index=False))


Ringkasan Status Stok 88 Barang:
status
AMAN       49
REORDER    39

Contoh 15 Barang dengan Status REORDER:
        sku            nama_barang    kategori satuan  stok_akhir  prediksi_permintaan_harian  safety_stock  reorder_point_rop  status
  UM-0300-M    Uzin SC 50 @40 Kg-M Barang Jadi    sak         174                       26.72           263                451 REORDER
  UM-0307-M    Uzin SC 75 @40 Kg-M Barang Jadi    sak         258                       20.31           143                286 REORDER
  UM-0310-M   Uzin SC 100 @40 Kg-M Barang Jadi    sak          67                       16.26            46                160 REORDER
  UM-0311-M Uzin SC 100 W @40 Kg-M Barang Jadi    sak           3                        1.10            81                 89 REORDER
  UM-0330-M    Uzin SC 300 @40Kg-M Barang Jadi    sak          53                       11.72            36                119 REORDER
    FG-TB20                  TB 20 Barang Jadi    sak         136                